# Belief State Existence Tests
This notebook empirically evaluates whether the residual streams of transformers trained on partially observed physical systems (*MessDriven* pendulum) genuinely encode a **belief state** (hidden HMM posterior distribution over environmental moods) rather than mere physics state or token identity artifacts.

### Converging Lines of Evidence:
1. **Test A: Null-Space Probing** — Does belief survive after projecting out the 2D physics subspace?
2. **Test B: Layer Emergence Curves** — Does belief emerge nonlinearly across depth, unlike physical observables?
3. **Test C: PCA Alignment** — Does training concentrate belief information into a low-dimensional subspace?
4. **Test D: Temporal Selectivity** — Does belief decodability change systematically across the within-cycle interval?
5. **Test G: UMAP Geometry** — Does the residual manifold mirror the belief simplex geometry?
6. **Test F: Matched-Physics Divergence** — When physical states match exactly but beliefs differ, does the residual stream diverge?


In [ ]:
import sys
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Put sweep dir and root on path
SWEEP_DIR = Path.cwd()
ROOT = SWEEP_DIR.parents[1] if (SWEEP_DIR / "existence_tests.py").exists() else Path.cwd()
if str(SWEEP_DIR) not in sys.path:
    sys.path.insert(0, str(SWEEP_DIR))

from existence_tests import (
    build_process_from_config,
    load_trained_and_random,
    generate_data,
    sequence_split,
    extract_features,
    extract_targets,
    test_nullspace,
    test_layer_emergence,
    test_pca_alignment,
    test_temporal_selectivity,
    test_matched_physics,
    test_umap_geometry,
    plot_multi_config,
    plot_nullspace,
    plot_layer_emergence,
    plot_pca_alignment,
    plot_temporal_selectivity,
    plot_matched_physics,
    plot_umap_geometry,
    belief_colors,
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


## Configuration & Hyperparameters
All knobs, model checkpoints, data sizes, seeds, and regularization strengths are defined here. No hidden constants.


In [ ]:
OUTPUT_DIR = SWEEP_DIR / "parallel_output"

# Configurations to test (ordered by predicted belief clarity)
CONFIGS = [
    "dt0.2_gamma1_dv1.2_n10",    # Sweet spot: highest belief gap (+0.50)
    "dt0.2_gamma0.8_dv1.2_n10",  # High gap (+0.50), high decorrelation
    "dt0.2_gamma0.8_dv1_n10",    # Robustness check (+0.45)
    "dt0.2_gamma0.3_dv0.7_n10",  # Negative control: physics-heavy (+0.30)
]

D_MODEL = 128
K_SUFFIX = "kn2"  # k = n_steps // 2 (lookahead horizon)
MODEL_SEED = 0

# Data sampling parameters
N_DATA = 1024
DATA_SEEDS = [42, 137, 2024]
TRAIN_FRAC = 0.8
RIDGE_ALPHA = 1.0
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Target Configs: {CONFIGS}")

# Output directory for saving figures
SAVE_FIG_DIR = SWEEP_DIR / "smoke_test_outputs"
SAVE_FIG_DIR.mkdir(parents=True, exist_ok=True)


## Load Processes, Checkpoints & Datasets
Constructs physics processes matching checkpoint metadata, loads trained transformers alongside untrained random baselines, and generates reproducible test trajectories across seeds.


In [ ]:
processes = {}
models = {}     # config -> {"trained": model, "random": model}
datasets = {}   # config -> {seed: data_dict}

for cfg in CONFIGS:
    cfg_dir = OUTPUT_DIR / cfg
    print(f"Initializing {cfg}...")
    proc = build_process_from_config(cfg, OUTPUT_DIR)
    processes[cfg] = proc
    
    tr_m, rnd_m = load_trained_and_random(cfg_dir, D_MODEL, K_SUFFIX, MODEL_SEED, DEVICE)
    models[cfg] = {"trained": tr_m, "random": rnd_m}
    
    datasets[cfg] = {}
    for s in DATA_SEEDS:
        datasets[cfg][s] = generate_data(proc, n=N_DATA, seed=s)

print("Initialization complete.")


## Test A: Null-Space Probing ("Belief Beyond Physics")
**Question**: If we mathematically remove the 2D physics subspace (theta, omega) from the residual stream, does belief information survive?
- We fit Ridge regression $W_\text{phys}$ on the residual stream.
- We project the residual stream onto the null space: $X_\text{null} = X - X Q Q^T$.
- We probe $X_\text{null}$ for belief.
- If $R^2_\text{null} > 0$ significantly above random baseline, belief is represented **orthogonally** to physics.


In [ ]:
NULLSPACE_MODES = ["all_layers_single_token", "all_layers_cycle"]
nullspace_results = {}

for mode in NULLSPACE_MODES:
    print(f"Running Test A (Null-Space Probing) in mode: {mode}...")
    nullspace_results[mode] = {}
    
    for cfg in CONFIGS:
        proc = processes[cfg]
        nullspace_results[mode][cfg] = {"trained": [], "random": []}
        
        for m_type in ["trained", "random"]:
            m = models[cfg][m_type]
            for s in DATA_SEEDS:
                data = datasets[cfg][s]
                res = test_nullspace(
                    model=m,
                    data=data,
                    proc=proc,
                    mode=mode,
                    alpha=RIDGE_ALPHA,
                    train_frac=TRAIN_FRAC,
                    device=DEVICE,
                    split_seed=s,
                    batch_size=BATCH_SIZE,
                )
                nullspace_results[mode][cfg][m_type].append(res)

print("Test A completed.")


### Plot: Test A Null-Space Probing Results


In [ ]:
for mode in NULLSPACE_MODES:
    fig, axes = plot_multi_config(
        nullspace_results[mode],
        plot_fn=plot_nullspace,
        n_plots_per_config=2,
        max_cols=2,
        figsize_per_subplot=(5.0, 3.2),
        title=f"Test A: Null-Space Probing (Mode: {mode})",
    )
    fig.savefig(SAVE_FIG_DIR / f"test_a_nullspace_{mode}.png", dpi=150, bbox_inches="tight")
    plt.show()


## Test B: Layer-wise Emergence Curves
**Question**: At which layer does belief become decodable?
- Physics is directly encoded in input tokens (theta), so physics $R^2$ is expected to be high at the embedding layer (Layer 0).
- Belief requires multi-step recursive temporal inference, so belief $R^2$ should emerge gradually across deeper layers.


In [ ]:
LAYER_VARIATIONS = ["last_token", "cycle_concat", "all_tokens"]
layer_results = {}

for cfg in CONFIGS:
    print(f"Running Test B (Layer Emergence) for {cfg}...")
    proc = processes[cfg]
    layer_results[cfg] = {"trained": {}, "random": {}}
    
    for m_type in ["trained", "random"]:
        m = models[cfg][m_type]
        for var in LAYER_VARIATIONS:
            layer_results[cfg][m_type][var] = []
            for s in DATA_SEEDS:
                data = datasets[cfg][s]
                res = test_layer_emergence(
                    model=m,
                    data=data,
                    proc=proc,
                    variation=var,
                    alpha=RIDGE_ALPHA,
                    train_frac=TRAIN_FRAC,
                    device=DEVICE,
                    split_seed=s,
                    batch_size=BATCH_SIZE,
                )
                layer_results[cfg][m_type][var].append(res)

print("Test B completed.")


### Plot: Test B Layer-wise Emergence


In [ ]:
fig, axes = plot_multi_config(
    layer_results,
    plot_fn=plot_layer_emergence,
    n_plots_per_config=2,
    max_cols=2,
    figsize_per_subplot=(5.2, 3.4),
    title="Test B: Layer-wise Emergence Curves (Trained vs Random)",
)
fig.savefig(SAVE_FIG_DIR / "test_b_layer_emergence.png", dpi=150, bbox_inches="tight")
plt.show()


## Test C: PCA Alignment ("Does Training Concentrate Belief?")
**Question**: In trained models, does belief information concentrate into the top principal components of the residual stream?


In [ ]:
PCA_FEATURE_CONFIGS = ["all_layers_cycle", "all_layers_single_token", "single_layer_all_tokens"]
PCA_MAX_PCS = 50
pca_results = {}

for cfg in CONFIGS:
    print(f"Running Test C (PCA Alignment) for {cfg}...")
    proc = processes[cfg]
    pca_results[cfg] = {"trained": {}, "random": {}}
    
    for m_type in ["trained", "random"]:
        m = models[cfg][m_type]
        for feat_cfg in PCA_FEATURE_CONFIGS:
            pca_results[cfg][m_type][feat_cfg] = []
            for s in DATA_SEEDS:
                data = datasets[cfg][s]
                res = test_pca_alignment(
                    model=m,
                    data=data,
                    proc=proc,
                    feature_config=feat_cfg,
                    alpha=RIDGE_ALPHA,
                    train_frac=TRAIN_FRAC,
                    device=DEVICE,
                    split_seed=s,
                    max_pcs=PCA_MAX_PCS,
                    batch_size=BATCH_SIZE,
                )
                pca_results[cfg][m_type][feat_cfg].append(res)

print("Test C completed.")


### Plot: Test C PCA Alignment Curves


In [ ]:
fig, axes = plot_multi_config(
    pca_results,
    plot_fn=plot_pca_alignment,
    n_plots_per_config=3,
    max_cols=3,
    figsize_per_subplot=(4.2, 3.2),
    title="Test C: PCA Alignment — Cumulative Belief R² vs Number of PCs",
)
fig.savefig(SAVE_FIG_DIR / "test_c_pca_alignment.png", dpi=150, bbox_inches="tight")
plt.show()


## Test D: Temporal Selectivity (Within-Cycle Gradient)
**Question**: How does decodability evolve across positions within a single cycle (t = 0 .. 9)?
- Note: Belief state updates only at cycle boundaries (kicks), so ground truth belief is constant within each cycle.
- Any change in belief $R^2$ across steps 0 .. 9 reflects active within-cycle representational dynamics.


In [ ]:
temporal_results = {}

for cfg in CONFIGS:
    print(f"Running Test D (Temporal Selectivity) for {cfg}...")
    proc = processes[cfg]
    m = models[cfg]["trained"]
    temporal_results[cfg] = []
    
    for s in DATA_SEEDS:
        data = datasets[cfg][s]
        res = test_temporal_selectivity(
            model=m,
            data=data,
            proc=proc,
            alpha=RIDGE_ALPHA,
            train_frac=TRAIN_FRAC,
            device=DEVICE,
            split_seed=s,
            batch_size=BATCH_SIZE,
        )
        temporal_results[cfg].append(res)

print("Test D completed.")


### Plot: Test D Within-Cycle Gradient


In [ ]:
fig, axes = plot_multi_config(
    temporal_results,
    plot_fn=plot_temporal_selectivity,
    n_plots_per_config=2,
    max_cols=2,
    figsize_per_subplot=(5.0, 3.2),
    title="Test D: Temporal Selectivity Across Cycle Positions (t=0..9)",
)
fig.savefig(SAVE_FIG_DIR / "test_d_temporal_selectivity.png", dpi=150, bbox_inches="tight")
plt.show()


## Test G: UMAP Geometry and Simplex Representation
**Question**: Does the low-dimensional manifold of the residual stream preserve the simplex geometry of the ground truth belief distribution?
- Points are colored by a weighted blend of the 4 mood colors:
  - Red: Mood 0, Blue: Mood 1, Green: Mood 2, Amber: Mood 3.
- Pure colors represent high confidence; blended colors represent mixed belief states.


In [ ]:
UMAP_FEATURE_CONFIGS = ["all_layers_cycle", "all_layers_single_token", "single_layer_all_tokens"]
UMAP_N_EPOCHS = 20
UMAP_MAX_POINTS = 5000
umap_results = {}

for cfg in CONFIGS:
    print(f"Running Test G (UMAP Geometry) for {cfg}...")
    proc = processes[cfg]
    m = models[cfg]["trained"]
    umap_results[cfg] = {}
    
    for feat_cfg in UMAP_FEATURE_CONFIGS:
        umap_results[cfg][feat_cfg] = []
        # Run on primary seed (or multiple)
        s = DATA_SEEDS[0]
        data = datasets[cfg][s]
        res = test_umap_geometry(
            model=m,
            data=data,
            proc=proc,
            feature_config=feat_cfg,
            n_epochs=UMAP_N_EPOCHS,
            n_neighbors=15,
            min_dist=0.1,
            seed=s,
            max_points=UMAP_MAX_POINTS,
            device=DEVICE,
            batch_size=BATCH_SIZE,
        )
        umap_results[cfg][feat_cfg].append(res)

print("Test G completed.")


### Plot: Test G UMAP Simplex Geometry


In [ ]:
fig, axes = plot_multi_config(
    umap_results,
    plot_fn=plot_umap_geometry,
    n_plots_per_config=3,
    max_cols=3,
    figsize_per_subplot=(4.2, 3.8),
    title="Test G: UMAP Manifold Colored by Belief Simplex Distribution",
)
fig.savefig(SAVE_FIG_DIR / "test_g_umap_geometry.png", dpi=150, bbox_inches="tight")
plt.show()


## Test F: Matched-Physics Divergence Analysis
**Question**: When physical state (theta, omega) is nearly identical (within epsilon) but hidden beliefs differ by > 0.3, does the residual stream diverge?
- We evaluate on late cycles (30 .. 45) where belief has settled.
- We generate a larger batch of trajectories (N = 2048) to ensure dense coverage of the physical phase space.


In [ ]:
MATCHED_N_DATA = 2048
MATCHED_CYCLE_RANGE = (30, 45)
MATCHED_EPSILON = 0.05
MATCHED_BELIEF_THRESHOLD = 0.3
matched_results = {}

for cfg in CONFIGS:
    print(f"Running Test F (Matched Physics) for {cfg} with N={MATCHED_N_DATA}...")
    proc = processes[cfg]
    m = models[cfg]["trained"]
    matched_results[cfg] = []
    
    for s in DATA_SEEDS[:2]:  # Use 2 seeds for large matched dataset
        fresh_data = generate_data(proc, n=MATCHED_N_DATA, seed=s + 1000)
        res = test_matched_physics(
            model=m,
            data=fresh_data,
            proc=proc,
            cycle_range=MATCHED_CYCLE_RANGE,
            epsilon=MATCHED_EPSILON,
            belief_threshold=MATCHED_BELIEF_THRESHOLD,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            random_seed=s,
        )
        matched_results[cfg].append(res)
        del fresh_data
        gc.collect()

print("Test F completed.")


### Plot: Test F Matched-Physics Residual Divergence


In [ ]:
fig, axes = plot_multi_config(
    matched_results,
    plot_fn=plot_matched_physics,
    n_plots_per_config=2,
    max_cols=2,
    figsize_per_subplot=(5.0, 3.2),
    title="Test F: Residual Stream Divergence for Matched Physics States",
)
fig.savefig(SAVE_FIG_DIR / "test_f_matched_physics.png", dpi=150, bbox_inches="tight")
plt.show()


## Overall Quantitative Scoreboard
Summary table aggregating all six tests across configurations.


In [ ]:
summary_rows = []

for cfg in CONFIGS:
    # Test A: Null belief R2
    tA_tr = np.mean([r["null_belief_r2"] for r in nullspace_results["all_layers_single_token"][cfg]["trained"]])
    tA_rnd = np.mean([r["null_belief_r2"] for r in nullspace_results["all_layers_single_token"][cfg]["random"]])
    null_gap = tA_tr - tA_rnd
    
    # Test B: Layer emergence delta (all - emb)
    runs_b = layer_results[cfg]["trained"]["last_token"]
    emb_r2 = np.mean([[d["belief_r2"] for d in run if d["layer"] == "emb"][0] for run in runs_b])
    all_r2 = np.mean([[d["belief_r2"] for d in run if d["layer"] == "all"][0] for run in runs_b])
    emergence_delta = all_r2 - emb_r2
    
    # Test C: Top 5 PCs belief R2
    pca_runs = pca_results[cfg]["trained"]["all_layers_single_token"]
    pca_5_r2 = np.mean([r["belief_r2"][min(3, len(r["belief_r2"])-1)] for r in pca_runs])
    
    # Test D: Temporal gradient (step 9 - step 0)
    temp_runs = temporal_results[cfg]
    temp_grad = np.mean([r["belief_r2"][-1] - r["belief_r2"][0] for r in temp_runs])
    
    # Test F: Matched separation ratio (matched / control)
    mf_runs = matched_results[cfg]
    m_l2 = np.mean([r["mean_matched_l2"] for r in mf_runs if not np.isnan(r["mean_matched_l2"])])
    c_l2 = np.mean([r["mean_control_l2"] for r in mf_runs if not np.isnan(r["mean_control_l2"])])
    ratio = m_l2 / c_l2 if c_l2 > 0 else np.nan
    
    summary_rows.append({
        "Configuration": cfg,
        "Null Belief R² (Trained)": f"{tA_tr:.3f}",
        "Null Belief Gap (Tr - Rnd)": f"{null_gap:+.3f}",
        "Layer Emergence (All - Emb)": f"{emergence_delta:+.3f}",
        "Top-5 PCs Belief R²": f"{pca_5_r2:.3f}",
        "Temporal Slope (t9 - t0)": f"{temp_grad:+.3f}",
        "Matched / Control L2": f"{ratio:.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)
